In [111]:
from typing import List, Dict
import random
import math
import json

class Engine:
    def __init__(self, features):
        """
        We have an equation which decides what percentage of the current cash to invest on that day.
        So one number per day which is decided by a matrix multiplication. 
        We have K market data metrics (AAPL prediction, MSFT prediction, ... diversification, interaction terms) for that day: features.
        We have a randomly initialized (or loaded from a file) set of betas.
        y=XB --> 1x1 = 1xK * Kx1 --> so beta should be a Kx1 random vector 
        buy_proportion_features should be a list of strings that correspond to the column names of the market data 


        The stock split will be:
        y=XB --> 1xS = 1xK * K*S where S is the number of stocks and K is the number of features. 

        """
        self.features = features

        self.create_weights()

    
    @classmethod
    def load_weights(cls, file_path):
        with open(file_path, "r", encoding="utf-8") as fp:
            self.params = json.load(fp)

    def create_weights(self):
        scale_1 = random.randint(1, 10000)
        scale_2 = random.randint(1, 10000)
        scale_3 = random.randint(1, 10000)
        scale_4 = random.randint(1, 10000)
        # Kx1 
        beta_buy = np.random.normal(loc=0, scale=scale_1, size=(len(self.features), 1))
        beta_sell = np.random.normal(loc=0, scale=scale_2, size=(len(self.features), 1))
        # KxS
        beta_buy_split = np.random.normal(loc=0, scale=scale_3, size=(len(self.features), len(self.stock_pool)))
        beta_sell_split = np.random.normal(loc=0, scale=scale_4, size=(len(self.features), len(self.stock_pool)))

    
    def mutate(self, other): 
        equal = (
            (set(self.buy_proportion_features) == set(other.buy_proportion_features))
            &
            (set(self.sell_proportion_features) == set(other.sell_proportion_features))
            &
            (set(self.buy_stock_split_features) == set(other.buy_stock_split_features))
            &
            (set(self.sell_stock_split_features) == set(other.sell_stock_split_features))
        )
        if not equal:
            raise ValueError('Features are not the same. Cannot mutate.')
        
        
        
        

    
    def mutate(self, other, show=True):
        if not set(self.params.keys()) == set(other.params.keys()):
            raise KeyError('Two engines do not have the same input keys. Cannot mutate.')
        else:
            mutated = {}
            for k in self.params.keys():
                avg = (self.params[k] + other.params[k]) / 2
                noise = random.gauss(mu=0, sigma=abs(self.params[k] - other.params[k])/3)
                mutated[k] = avg + noise

                if show:
                    print('Noise: ')
                    print(noise)

            if show:
                print('Before: ')
                print(self.params)
                print(other.params)
                print('After: ')
                print(mutated)
            return mutated


In [142]:
e1 = Engine({'a': random.random(), 'b': random.randint(1, 10)})
e2 = Engine({'a': random.random(), 'b': random.randint(100, 200)})
e1.mutate(e2)

Noise: 
-0.9153760372917946
Noise: 
83.15539536705035
Before: 
{'a': 0.08077451228964949, 'b': 6}
{'a': 0.9576834521564211, 'b': 160}
After: 
{'a': -0.39614705506875925, 'b': 166.15539536705035}


{'a': -0.39614705506875925, 'b': 166.15539536705035}

In [194]:
import numpy as np

In [201]:
(np.array([1,2,3]) + np.array([4,5,6]) )/2

array([2.5, 3.5, 4.5])

In [ ]:
"""
10 days
AAPL -> +0.15 , forecast, posession dummy
MSFT -> -0.10 , 
AMZN -> +0.95 , 

y = XB
10x3 = 10x6 * 6x3

In [191]:
import numpy as np
from sklearn.linear_model import LinearRegression

def softmax(arr):
    shifted = arr - arr.max(axis=1, keepdims=True)
    exp_vals = np.exp(shifted)
    weights = exp_vals / exp_vals.sum(axis=1, keepdims=True)
    return weights

def sigmoid(arr):
    return 1/(1+np.exp(-arr))

# How much to invest today

X = np.random.standard_normal((10, 6)) # this would come from the dataset (10 days x 2 features for 3 stocks)
beta_1 = np.full((6, 1), [random.random()]) # this is initialized randomly in the engine --> params should be a numpy array and not a dict
beta_2 = np.full((6,3), np.random.standard_normal((1, 3)))  # this is initialized randomly in the engine --> params should be a numpy array and not a dict

spending_proportion = sigmoid(X.dot(beta_1))
split_across_stocks = softmax(X.dot(beta_2))

spending_proportion, split_across_stocks

(array([[0.49755486],
        [0.48470772],
        [0.27651143],
        [0.51772705],
        [0.38275841],
        [0.49027432],
        [0.40122961],
        [0.78266096],
        [0.10069278],
        [0.41289518]]),
 array([[3.26689571e-01, 3.30420487e-01, 3.42889942e-01],
        [2.91778880e-01, 3.13261353e-01, 3.94959767e-01],
        [8.28133711e-03, 2.52979864e-02, 9.66420676e-01],
        [3.80967039e-01, 3.50847445e-01, 2.68185515e-01],
        [7.47273329e-02, 1.30145567e-01, 7.95127100e-01],
        [3.06887278e-01, 3.21068206e-01, 3.72044515e-01],
        [1.01601668e-01, 1.61721116e-01, 7.36677216e-01],
        [8.14540386e-01, 1.84022772e-01, 1.43684177e-03],
        [1.96942409e-05, 2.50252153e-04, 9.99730054e-01],
        [1.21757109e-01, 1.83226536e-01, 6.95016355e-01]]))

In [193]:
beta_2

array([[ 1.1599564 ,  0.65280663, -1.00157613],
       [ 1.1599564 ,  0.65280663, -1.00157613],
       [ 1.1599564 ,  0.65280663, -1.00157613],
       [ 1.1599564 ,  0.65280663, -1.00157613],
       [ 1.1599564 ,  0.65280663, -1.00157613],
       [ 1.1599564 ,  0.65280663, -1.00157613]])